# First Linear Checkpoint: Linear Modeling Baseline

This notebook is the educational entrypoint for the first modeling-stage checkpoint. It runs the reproducible script pipeline, then walks through the research design, chronological split, feature-only main model, historical persistence baselines, a persistence-augmented comparison model, missingness diagnostics, rolling-origin confirmation, robustness checks, mechanism submodels, and coefficient interpretation.

Teaching scaffold used throughout: **Question -> Concept -> Output -> How to read -> Takeaway**. The notebook should explain how to read each result, while reusable modeling and verification logic stays in `3_models/scripts/`.


## Learning Goals

By the end of this notebook, you should be able to explain:

1. Why this is a **First Linear Checkpoint**, not the final forecasting claim.
2. Why the primary model is feature-only rather than silently mixing lagged target history into the main predictor set.
3. How the teacher-confirmed chronological 80/10/10 split avoids random row-level time leakage.
4. Why historical target baselines are necessary before making any forecasting claim.
5. How to read `history_delta_summary`: positive delta MAE means the feature-only model trails a prediction-safe historical baseline.
6. How the confirmatory rolling-origin check distributes held-out blocks through time while keeping validation inside each pre-test window.
7. Why predictor missingness is preserved in the input, imputed inside the sklearn pipeline, and tested with both complete-case and `missingness_indicator_refit` sensitivity.
8. How year-level and target-quantile error decomposition show whether aggregate MAE hides high-innovation-country failures.
9. How to read MAE, RMSE, out-of-sample R2, Spearman rank correlation, robustness checks, and standardized coefficients without turning prediction results into causal claims.

Professor-level guardrail: **do not claim that external predictors beat national historical persistence** unless the feature-only model beats prediction-safe country-history baselines on the relevant evaluation design.


## Optional Colab Setup

In [ ]:
# Local runs skip this setup branch.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('Clone or upload the repository, then set PROJECT_ROOT = Path("/content/env-innovation-prediction") before continuing.')
else:
    print('Local run detected; setup skipped.')

## Run the Reproducible Pipeline

The notebook does not duplicate modeling logic. It imports `run_modeling.py`, executes the same pipeline that can be run from the terminal, and then displays the generated tables and figures.

In [ ]:
from pathlib import Path
import sys
import time

import pandas as pd
from IPython.display import Image, Markdown, display

try:
    PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
except NameError:
    PROJECT_ROOT = Path.cwd().resolve()
    while PROJECT_ROOT.name != 'env-innovation-prediction' and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / '3_models' / 'scripts' / 'run_modeling.py').exists():
    raise FileNotFoundError(f'Could not find modeling scripts under PROJECT_ROOT={PROJECT_ROOT}')

SCRIPT_DIR = PROJECT_ROOT / '3_models' / 'scripts'
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

from run_modeling import run_linear_modeling, verify_generated_artifacts

run_started_at = time.time()
results = run_linear_modeling()
figure_paths = results['figure_paths']

def show_figure(name: str) -> None:
    display(Image(filename=figure_paths[name]))

display(Markdown(f"**Best validation model:** `{results['best_model']}`"))


## Step 1: Data Contract and Specification Registry

**Question -> Concept -> Output -> How to read -> Takeaway**

**Question.** What exactly is being modeled, and which protocol did we commit to before reading test results?

**Concept.** The first baseline uses the active v2 no-imputation main panel. The primary target is `env_patent_share_inventions`, and the primary features are the nine `*_lag1_3_mean` predictors. The panel already drops rows with missing target values; predictor missingness is intentionally preserved so that imputation can be fit inside the training pipeline rather than before the split.

**Output.** The sample summary and specification registry below are the audit trail: target, lag scheme, sample rule, imputation rule, model family, and validation selection metric.

**How to read.** Start with `split`, `year_start`, `year_end`, `rows`, `countries`, `imputation_rule`, and `selection_metric`. These columns tell you what evidence each later table is allowed to support.

**Takeaway.** This is a reproducible first checkpoint with a predeclared feature-only main specification, not a post-test search for the best story.


In [ ]:
display(results['sample_summary'])
display(results['specification_registry'])
pd.DataFrame({'feature': results['feature_columns']})


## Step 2: Chronological 80/10/10 Split

The split is made over distinct target years, not shuffled rows. For the 1999-2023 panel this gives training years 1999-2018, validation years 2019-2020, and test years 2021-2023. The final test block is the latest period and is not used to choose the model.

In [ ]:
show_figure('split_rows')

## Step 3: Target Scale and Research Question

The target is right-skewed, so large innovation countries can dominate squared-error metrics. For this checkpoint we keep the target in original units because MAE remains interpretable. The more important design choice is conceptual: the main model is a feature-only main model, while target history is evaluated separately as a baseline and then as an explicit augmented comparison.


In [ ]:
main_panel = results['experiments']['main']['panel']
target_column = results['experiments']['main']['target_column']
target_summary = main_panel[target_column].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).to_frame('target')
display(target_summary)


## Step 4: Predictor Missingness and Train-Fold Imputation

The saved panel is a no-imputation panel. Missing predictor values remain as `NaN` until the sklearn pipeline is fit. Inside each candidate model, median imputation and scaling are fit on the training fold only, then reused for validation. After model selection, the final pipeline is refit on train plus validation before scoring the test block. The linear-interpolated panels are retrospective coverage sensitivity outputs only; they are not used in this modeling run.

The first figure shows the overall missing share by predictor among target-observed model rows. The next diagnostic goes further: it asks whether missing values in those rows mostly come from late-start coverage, early-ending series, bounded coverage windows, intermittent gaps, or countries with no observed values for a given predictor. Those patterns imply different interpretation risks.


In [ ]:
show_figure('feature_missingness')

## Step 5: Missingness Pattern Diagnostics

A single missing-share number can hide very different data-generating situations. These diagnostics classify each country-predictor sequence among target-observed model rows, not a fully balanced country-calendar-year grid. `total_years` counts rows present in the supervised model panel, while `calendar_year_span` records the span between the first and last target-observed row for that country.

Late-start coverage means observed predictor values begin after the country's early model rows. Early-ending coverage means observed values stop before the country's recent model rows. Bounded windows have both leading and trailing missing rows. Intermittent gaps mean observations disappear inside an otherwise covered country series; this category takes priority when a sequence also has leading or trailing missing rows. All-missing country-feature histories mean that the model can only use the train-fold median for that predictor-country combination.

This table and figure do not change the baseline model. They document why train-only median imputation is conservative and where future robustness checks, such as past-only carry-forward or source-specific exclusions, would be most relevant.


In [ ]:
missingness_pattern_summary = results['missingness_pattern_summary']
missingness_pattern_detail = results['missingness_pattern_country_detail']

pattern_columns = [
    'feature',
    'missing_share',
    'complete_countries',
    'late_start_countries',
    'early_end_countries',
    'bounded_coverage_window_countries',
    'intermittent_gaps_countries',
    'all_missing_countries',
]
display(missingness_pattern_summary[pattern_columns])
show_figure('missingness_pattern_counts')

diagnostic_examples = (
    missingness_pattern_detail[
        missingness_pattern_detail['missingness_pattern'].isin(
            ['late_start', 'intermittent_gaps', 'bounded_coverage_window', 'all_missing']
        )
    ]
    .sort_values(['missing_years', 'feature', 'country_code'], ascending=[False, True, True])
    .head(15)
)
display(
    diagnostic_examples[
        [
            'feature',
            'country_code',
            'country_name',
            'missingness_pattern',
            'year_start',
            'year_end',
            'calendar_year_span',
            'total_years',
            'first_observed_year',
            'last_observed_year',
            'missing_years',
            'internal_missing_years',
        ]
    ]
)


## Step 6: Validation Chooses the Linear Candidate

**Question -> Concept -> Output -> How to read -> Takeaway**

**Question.** Which linear candidate is selected without looking at the final test block?

**Concept.** The training period fits each candidate linear model. The validation period chooses the candidate with the lowest MAE. This keeps the final test block reserved for one final evaluation after model choice.

**Output.** The table and figure rank linear candidates by validation MAE.

**How to read.** The smallest validation MAE wins; RMSE is a tie-break and a large-error warning, not the primary selector.

**Takeaway.** Selection is validation-driven, so later test and rolling-origin results can be interpreted as held-out checks rather than tuning feedback.


In [ ]:
display(results['validation_metrics'].head(10))
show_figure('validation_mae')

## Step 7: Final Test Evaluation

**Question -> Concept -> Output -> How to read -> Takeaway**

**Question.** How does the selected first-checkpoint linear model perform on the latest held-out block?

**Concept.** After validation selects the model, the selected pipeline is refit on train plus validation years and evaluated on 2021-2023. MAE is the headline metric because it is in target units. RMSE highlights large misses, out-of-sample R2 compares against the training-period mean, and Spearman checks rank ordering.

**Output.** The test table reports aggregate metrics, and the scatter plot shows observed versus predicted values.

**How to read.** A decent OOS R2 or Spearman does not prove the model beats country persistence; that question is answered by the historical-baseline delta table next.

**Takeaway.** Treat this as a first late-period stress check, not as the final proof of forecasting superiority.


In [ ]:
display(results['test_metrics'])
show_figure('actual_vs_predicted')

## Step 8: Persistence Baselines and History Delta

**Question -> Concept -> Output -> How to read -> Takeaway**

**Question.** Does the feature-only model beat simple prediction-safe target-history rules?

**Concept.** The strongest reviewer concern is whether the feature model improves on simple target-history forecasts. The baselines below use only target information available before the held-out test block: a global train+validation mean, a country train+validation mean, and a country last-pretest value held constant through 2021-2023.

**Output.** The first table shows baseline metrics. The `history_delta_summary` table then subtracts each historical baseline MAE from the selected feature-only model MAE.

**How to read.** Positive `delta_mae_selected_minus_baseline` means the feature-only model is worse than that historical baseline. Negative values would be needed before claiming that external predictors beat national historical persistence.

**Takeaway.** In this checkpoint, the narrative must remain conservative: the feature-only model is useful for interpretable associations and ranking diagnostics, not for claiming superiority over country history.


In [ ]:
historical_baselines = results['historical_baselines']
history_delta_summary = results['history_delta_summary']
display(historical_baselines)
display(history_delta_summary)
show_figure('historical_baseline_mae')

selected_mae = history_delta_summary['selected_mae'].iloc[0]
best_delta = history_delta_summary.sort_values('baseline_mae').iloc[0]
display(Markdown(
    f"Feature-only test MAE is **{selected_mae:.3f}**. "
    f"Against `{best_delta['baseline_model']}`, delta MAE is "
    f"**{best_delta['delta_mae_selected_minus_baseline']:.3f}**. "
    "Positive delta means the feature-only model trails the history baseline, so do not claim that external predictors beat national historical persistence."
))


## Step 9: Persistence-Augmented Model

Now we answer the question directly: what happens if past target history is added as a predictor? We keep the primary model feature-only, then run a separate augmented comparison using `target_history_preblock`. This is a block-safe target history feature: validation years use only training-period target history, and test years use only train+validation target history. It never uses test-period labels inside the test block.

Read this as a decomposition: history-only shows national inertia; feature-only shows the external predictor baseline; feature plus history asks whether external predictors still matter after a persistence signal is available.


In [ ]:
persistence_augmented_comparison = results['persistence_augmented_comparison']
display(
    persistence_augmented_comparison[
        [
            'model_stage',
            'model',
            'includes_main_predictors',
            'includes_target_history',
            'validation_mae',
            'test_mae',
            'test_rmse',
            'test_oos_r2_vs_train_mean',
            'test_spearman',
        ]
    ]
)

augmented_coefficients = results['persistence_augmented_coefficients']
display(
    augmented_coefficients[
        ['model', 'feature', 'coefficient', 'abs_coefficient']
    ].sort_values('abs_coefficient', ascending=False).head(8)
)

history_only = persistence_augmented_comparison[
    persistence_augmented_comparison['model_stage'].eq('history_only_baseline')
].iloc[0]
augmented = persistence_augmented_comparison[
    persistence_augmented_comparison['model_stage'].eq('persistence_augmented_linear')
].iloc[0]
feature_only = persistence_augmented_comparison[
    persistence_augmented_comparison['model_stage'].eq('feature_only_linear')
].iloc[0]
display(Markdown(
    f"Feature-only test MAE is **{feature_only['test_mae']:.3f}**; "
    f"persistence-augmented test MAE is **{augmented['test_mae']:.3f}**; "
    f"history-only persistence baseline MAE is **{history_only['test_mae']:.3f}**. "
    "This shows that target history is highly predictive, and that adding it changes the research question rather than simply improving the original main model."
))


## Step 10: Failure Modes

After the persistence comparison, inspect where the feature-only model misses most. The table below lists the largest absolute test errors so limitations are visible rather than hidden behind aggregate metrics.


In [ ]:
top_errors = results['top_errors'].head(10)
display(top_errors)
show_figure('top_absolute_errors')


## Step 10b: Error Decomposition by Year and Target Scale

**Question -> Concept -> Output -> How to read -> Takeaway**

**Question.** Are aggregate errors hiding specific years or high-target country-years where the model fails?

**Concept.** A single MAE can look acceptable while errors concentrate in recent years or in high-innovation observations. The script therefore writes year-level and target-quantile error summaries.

**Output.** `error_by_year` groups errors by test year. `error_by_target_quantile` groups the same test predictions by observed target quantile.

**How to read.** Compare MAE and RMSE across rows. Higher target quantiles with larger MAE mean the model struggles where environmental innovation shares are largest.

**Takeaway.** This is a failure-mode diagnostic, not a post-test rule for deleting countries or retuning the model.


In [ ]:
display(results['error_by_year'])
display(results['error_by_target_quantile'])


## Step 11: Missingness Sensitivity

**Question -> Concept -> Output -> How to read -> Takeaway**

**Question.** Are conclusions fragile because many predictor values are missing?

**Concept.** The main panel is a no-imputation source panel, but the model itself uses train-fold median imputation inside the sklearn pipeline. The complete-case rerun checks what happens after removing rows with any missing active predictor. The `missingness_indicator_refit` keeps all rows and adds one binary missingness indicator per active predictor, preserving the original train-fold imputation pipeline.

**Output.** The sensitivity table compares primary median-imputed, complete-case, and missingness-indicator runs. The indicator plan shows which binary features were added.

**How to read.** Compare rows, countries, test rows, and test MAE together. A lower MAE with far fewer rows is not automatically a better main model.

**Takeaway.** Missingness remains a substantive limitation; these checks qualify the interpretation rather than replacing the primary feature-only specification.


In [ ]:
missingness_sensitivity = results['missingness_sensitivity']
display(missingness_sensitivity)
display(results['missingness_indicator_plan'])
show_figure('missingness_sensitivity_mae')


## Step 12: Linear Robustness Pack

Robustness here means checking whether the same qualitative conclusion survives a small, protocol-bounded set of sensitivity settings. We keep the teacher-confirmed chronological 80/10/10 split and the same validation-only model-selection rule, then vary three design dimensions: **common-sample lag1 versus lag1_3_mean** predictor timing, skew-transformed main predictors, and the alternative target `env_patents_per_million`.

This section is intentionally still linear. The candidate family now includes ordinary least squares, Ridge, Lasso, and ElasticNet. The finite regularization grid is shown in the specification registry. Predictors are standardized inside each pipeline, but the target is not standardized, so alpha values are target-scale dependent and should be read as a limited exploratory grid rather than a fully tuned regularization protocol.

The lag1 rows are common-sample timing sensitivities: they reuse the same generated panels and change only which lagged predictor columns are selected. The main scientific question is not whether one row has the smallest raw MAE across different target units. The question is whether the selected linear feature model beats the strongest prediction-safe historical baseline within each target setting.


In [ ]:
robustness_summary = results['robustness_summary']
robustness_columns = [
    'robustness_id',
    'target_column',
    'lag_suffix',
    'best_model',
    'rows',
    'countries',
    'validation_mae',
    'test_mae',
    'best_historical_baseline_model',
    'best_historical_baseline_mae',
    'delta_test_mae_selected_minus_best_history',
    'beats_best_historical_baseline',
]
display(robustness_summary[robustness_columns])
show_figure('robustness_validation_test_mae')
show_figure('robustness_history_delta')

beats_history = int(robustness_summary['beats_best_historical_baseline'].sum())
total_settings = len(robustness_summary)
display(Markdown(
    f"Robustness result: `{beats_history}` of `{total_settings}` protocol-bounded linear settings "
    "beat their strongest prediction-safe historical baseline. "
    "This supports reporting the feature models as interpretable association/ranking diagnostics rather than "
    "as superior short-horizon forecasts over national historical persistence."
))


## Step 13: Skew-Transformed Linear Robustness

Standard scaling puts predictors on comparable units, but it does not make highly right-skewed variables approximately normal. This robustness check asks whether the feature-only linear model was mainly hurt by long-tailed predictor scales such as GDP, scientific articles, FDI, trade openness, CO2 per capita, environmental technology RTA, and inflation.

The transformations are deterministic and use no fitted information from validation or test labels. Positive right-skewed variables use `log1p`; signed variables with negative values use `asinh`; bounded or already near-symmetric variables stay on their original scale before the same median-imputation and StandardScaler pipeline. This makes the check a linear-model robustness step before moving to nonlinear models, not a replacement for the primary feature-only baseline.


In [ ]:
skew_transform_plan = results['skew_transform_plan']
display(skew_transform_plan)

skew_id = 'robust_main_share_skew_transformed_lag1_3_mean'
primary_id = 'robust_main_share_lag1_3_mean'
skew_comparison = robustness_summary[
    robustness_summary['robustness_id'].isin([primary_id, skew_id])
].loc[:, [
    'robustness_id',
    'best_model',
    'validation_mae',
    'test_mae',
    'test_oos_r2_vs_train_mean',
    'test_spearman',
    'best_historical_baseline_model',
    'best_historical_baseline_mae',
    'beats_best_historical_baseline',
]]
display(skew_comparison)

primary_row = robustness_summary[robustness_summary['robustness_id'].eq(primary_id)].iloc[0]
skew_row = robustness_summary[robustness_summary['robustness_id'].eq(skew_id)].iloc[0]
display(Markdown(
    f"Skew-transform result: validation MAE changes from `{primary_row['validation_mae']:.3f}` "
    f"to `{skew_row['validation_mae']:.3f}`, and test MAE changes from `{primary_row['test_mae']:.3f}` "
    f"to `{skew_row['test_mae']:.3f}`. In this run, skew transformation does not rescue the "
    "feature-only linear model, so the next defensible step is to test nonlinear models as a separate "
    "model-family robustness stage rather than to relabel the transformed linear model as primary."
))


## Step 13b: Confirmatory Rolling-Origin Check

**Question -> Concept -> Output -> How to read -> Takeaway**

**Question.** Does the feature-only linear model behave similarly when held-out periods are distributed through the timeline?

**Concept.** The 80/10/10 split is the teacher-confirmed first checkpoint. A confirmatory rolling-origin check is stronger forecasting evidence because each fold trains on earlier years, validates inside the pre-test window, and tests the next held-out block.

**Output.** `rolling_origin_summary` reports fold-level validation MAE, test MAE, best historical baseline, and delta against history. `rolling_origin_predictions` is saved for audit and downstream plots.

**How to read.** Look first at `delta_mae_selected_minus_best_history` and `beats_best_historical_baseline` across folds. One good fold is not enough; the professor-level question is whether the pattern is stable.

**Takeaway.** This section separates a course-friendly first checkpoint from stronger confirmatory rolling-origin evidence and reduces the risk of over-reading one latest-period test block.


In [ ]:
rolling_origin_summary = results['rolling_origin_summary']
display(rolling_origin_summary)

folds_beating_history = int(rolling_origin_summary['beats_best_historical_baseline'].sum())
total_folds = len(rolling_origin_summary)
display(Markdown(
    f"Rolling-origin result: `{folds_beating_history}` of `{total_folds}` folds beat their strongest prediction-safe historical baseline. "
    "Use this as confirmatory evidence, not as a new hyperparameter search."
))


## Step 14: Feature-Only Coefficient Interpretation

The selected Elastic Net model is still a linear model. Coefficients are shown after median imputation and standard scaling inside the pipeline, so larger absolute values indicate stronger model weight on the standardized feature scale. These are predictive associations, not causal effects, and they must be interpreted together with historical baselines and feature collinearity.


In [ ]:
display(results['coefficients'])
show_figure('coefficients')

## Step 15: Main Model Versus Mechanism Submodels

The project requirement asks for interpretable prediction and substantive interpretation. The main v2 model remains the primary specification because it has the broadest balanced set of literature-backed predictors. The mechanism submodels ask narrower questions: whether RISE energy regulation, R&D/co-invention/tax capacity, or OECD EPS carry predictive signal in their own available panels.

The comparison below is an own-sample diagnostic. It is useful for understanding mechanisms, but it is not a direct ranking because the submodel panels have different countries and test years.


In [ ]:
panel_comparison = results['panel_comparison']
display(panel_comparison)
show_figure('panel_sample_comparison')
show_figure('test_metric_comparison')


## Step 16: Same-Sample Nested Submodel Tests

The pure submodel table above is useful descriptively, but it is not the main comparability test. For comparability, each submodel sample is evaluated twice on exactly the same country-year rows: first with only the main-model predictors, then with the main predictors plus the submodel predictors. The delta column below is the augmented model's test MAE minus the same-sample main-controls baseline; negative values mean the submodel variables improve MAE.

In the current run, none of the submodel augmentations improves primary test MAE. This is a negative result for incremental predictive value, not evidence that the mechanisms are unimportant causally.


In [ ]:
nested_comparison = results['nested_comparison']
display(
    nested_comparison[[
        'comparison_group',
        'rows',
        'countries',
        'baseline_validation_mae',
        'augmented_validation_mae',
        'delta_validation_mae_augmented_minus_baseline',
        'baseline_test_mae',
        'augmented_test_mae',
        'delta_test_mae_augmented_minus_baseline',
        'improves_primary_test_mae',
    ]]
)
show_figure('nested_test_mae_comparison')
show_figure('nested_test_mae_delta')


## Step 17: Compact Submodel Coefficient Check

Submodel coefficients are a secondary diagnostic. To keep the notebook focused, we show a compact table rather than another coefficient figure. A nonzero coefficient means that the narrow mechanism panel used that feature for prediction, not that the feature causally caused later environmental patenting.


In [ ]:
display(
    results['panel_coefficients']
    .sort_values(['panel_id', 'abs_coefficient'], ascending=[True, False])
    .groupby('panel_id', as_index=False, group_keys=False)
    .head(4)
    .loc[:, ['panel_id', 'panel_label', 'model', 'feature', 'coefficient', 'abs_coefficient']]
)
show_figure('panel_validation_mae')


## Step 18: ElasticNet Under Correlated Predictors

ElasticNet is useful here because country-level predictors are correlated. The diagnostics use train plus validation rows only, so the final test labels do not influence interpretation. The correlation tables are computed on the same imputed and scaled train+validation design matrix used by the fitted pipeline when available. Zero ElasticNet coefficients are marked as `shrunk_to_zero` rather than treated as sign-aligned.

These tables explain coefficient stability and overlap among predictors; they are not a post-test rule for dropping variables.


In [ ]:
diagnostics = results['correlation_diagnostics']
display(Markdown(
    f"Correlation diagnostics use train+validation years "
    f"{diagnostics['analysis_year_start']}-{diagnostics['analysis_year_end']} "
    f"with n={diagnostics['analysis_rows']} rows. "
    f"Design: `{diagnostics['correlation_design']}`."
))
display(diagnostics['top_correlated_pairs'])
display(diagnostics['target_correlations'])
display(diagnostics['coefficient_correlation_alignment'])
show_figure('main_feature_correlation_heatmap')
show_figure('main_target_correlations')
show_figure('main_coefficient_correlation_alignment')


## Output Files

The script writes reusable CSV outputs under `3_models/outputs/` and figure files under `4_analysis/figures/modeling/`. These files include the main linear model, historical baselines, failure-mode errors, missingness sensitivity, pure submodel diagnostics, same-sample nested tests, and correlation diagnostics.


In [ ]:
pd.DataFrame(
    [{'artifact': name, 'path': str(path)} for name, path in results['outputs'].items()]
)

## Artifact Completeness Check

This final cell makes the notebook self-checking without embedding verification logic in the notebook. It calls `verify_generated_artifacts()` from `run_modeling.py`, so artifact validation is reusable from tests or terminal runs as well.


In [ ]:
artifact_check = verify_generated_artifacts(
    results=results,
    run_started_at=run_started_at,
    project_root=PROJECT_ROOT,
)
artifact_count_summary = artifact_check['artifact_count_summary']
artifact_suffix_summary = artifact_check['artifact_suffix_summary']
display(artifact_count_summary)
display(artifact_suffix_summary)
display(Markdown(
    f"Generated artifact check passed: {int(artifact_count_summary.loc[artifact_count_summary['check'].eq('verified_files'), 'count'].iloc[0])} files verified."
))
